# One-dimensional $S_N$ criticality solver

This notebook introduces a compact Python implementation of the one-dimensional $S_N$ method. The same `SNSolver` class treats one-group and multigroup slab problems.

The numerical nesting is intentionally visible for teaching purposes:

1. **Power iterations** update the fission source and $k_\mathrm{eff}$ (in case of criticality calculations).
2. **Outer iterations** converge scattering between energy groups (no iterations if G=1).
3. **Inner iterations** converge within-group scattering.
4. Each inner iteration performs positive- and negative-direction **diamond-difference sweeps**.

Only vacuum-boundary criticality calculations are included here;

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# Locate the exercise directory both when Jupyter starts here and from a parent folder.
working_directory = Path.cwd().resolve()
candidates = (
    working_directory,
    *working_directory.parents,
    working_directory / 'Ex_RP_SN_discrete_ordinates',
    working_directory / 'RPTT_PoliTO_GitHub' / 'Ex_RP_SN_discrete_ordinates',
)
PROJECT_DIRECTORY = next(
    (path for path in candidates if (path / 'sn_1d.py').is_file()),
    None,
)
if PROJECT_DIRECTORY is None:
    raise FileNotFoundError('Cannot locate Ex_RP_SN_discrete_ordinates/sn_1d.py.')

sys.path.insert(0, str(PROJECT_DIRECTORY))
from sn_1d import SNSettings, SNSolver

DATA_DIRECTORY = PROJECT_DIRECTORY / 'data'
required_inputs = {
    'geometry_homogeneous.txt', 'geometry_reflected.txt', 'geometry_5regions.txt',
    'nucl_prop_homogeneous.json', 'nucl_prop_reflected.json',
    'nucl_prop_5regions.json', 'nucl_prop_MG_homogeneous.json',
    'nucl_prop_MG_reflected.json',
}
missing_inputs = sorted(name for name in required_inputs if not (DATA_DIRECTORY / name).is_file())
if missing_inputs:
    raise FileNotFoundError(f'Missing input files in {DATA_DIRECTORY}: {missing_inputs}')

print(f'Input directory: {DATA_DIRECTORY}')

plt.rcParams.update({'figure.figsize': (7, 4), 'axes.grid': True})

## One-group calculation

The settings below define the angular order and the convergence tolerances. `source_ratio` selects the standard fission-source update. Material data are read from JSON files containing named materials and their ordering across the slab. Change `order`, the tolerances, or the case name and rerun the cells.

In [ ]:
one_group_settings = SNSettings(
    order=2,
    inner_tolerance=1.0e-6,
    eigenvalue_tolerance=1.0e-6,
    eigenvalue_update='source_ratio',
)

one_group_solver = SNSolver.from_case(
    DATA_DIRECTORY,
    case='homogeneous',       # also try 'reflected' or '5regions'
    multigroup=False,
    settings=one_group_settings,
)
one_group_result = one_group_solver.solve_critical()

print(f'k_eff = {one_group_result.keff:.8f}')
print(f'power iterations = {one_group_result.power_iterations}')
print(f'converged = {one_group_result.converged}')
print(f'region materials = {one_group_result.problem.cross_sections.region_materials}')

In [ ]:
mesh = one_group_result.problem.mesh
normalised_flux = one_group_result.normalized_scalar_flux()[0]

fig, ax = plt.subplots()
ax.plot(mesh.cell_centers, normalised_flux, lw=2, label='$S_2$ transport')
for interface in mesh.region_edges[1:-1]:
    ax.axvline(interface, color='0.7', ls='--', lw=1)
ax.set(xlabel='$z$', ylabel='normalised scalar flux', title=f'$k_{{eff}}={one_group_result.keff:.7f}$')
ax.legend();

### Power- and source-iteration histories

Every result retains the full nested history. For a one-group problem, each power iteration contains one group pass and that pass contains one source-iteration history.

In [ ]:
power_number = np.array([item.iteration for item in one_group_result.history])
keff_error = np.array([item.keff_error for item in one_group_result.history])
flux_l2_error = np.array([item.flux_l2_error for item in one_group_result.history])
source_counts = np.array([
                            item.outer_iterations[0].inner_iterations[0].iterations
                            for item in one_group_result.history
                        ])

fig, (ax_error, ax_count) = plt.subplots(2, 1, sharex=True, figsize=(7, 7))
ax_error.semilogy(power_number, keff_error, label='$k_{eff}$ error')
ax_error.semilogy(power_number, flux_l2_error, label='flux $L_2$ error')
ax_error.set_ylabel('relative error')
ax_error.legend()
ax_count.plot(power_number, source_counts, 'o-', ms=3)
ax_count.set(xlabel='power iteration', ylabel='inner iterations');

## Alternative eigenvalue update

A Rayleigh-like eigenvalue update can be selected through the settings, without requiring a second solver implementation. In this teaching code it is provided only for one-group problems.

In [ ]:
rayleigh_settings = SNSettings(
    order=2,
    inner_tolerance=1.0e-6,
    eigenvalue_tolerance=1.0e-6,
    eigenvalue_update='rayleigh',
)
rayleigh_result = SNSolver.from_case(
    DATA_DIRECTORY, 'homogeneous', multigroup=False, settings=rayleigh_settings
).solve_critical()

print(f"source-ratio update: k_eff={one_group_result.keff:.8f}, "
      f"{one_group_result.power_iterations} power iterations")
print(f"Rayleigh-like update: k_eff={rayleigh_result.keff:.8f}, "
      f"{rayleigh_result.power_iterations} power iterations")

## Changing the angular order

This is one example of running the same model class with different parameters. Each call reloads the same physical problem and changes only the quadrature order.

In [ ]:
order_results = {}
for order in (2, 4, 8):
    settings = SNSettings(order=order, inner_tolerance=1e-6, eigenvalue_tolerance=1e-6)
    order_results[order] = SNSolver.from_case(
        DATA_DIRECTORY, 'homogeneous', multigroup=False, settings=settings
    ).solve_critical()

fig, ax = plt.subplots()
for order, result in order_results.items():
    ax.plot(result.problem.mesh.cell_centers, result.normalized_scalar_flux()[0],
            label=f'$S_{order}$: $k_{{eff}}={result.keff:.6f}$')
ax.set(xlabel='$z$', ylabel='normalised scalar flux', title='Angular-order comparison')
ax.legend();

## Multigroup calculation

Set `multigroup=True` to read `nucl_prop_MG_<case>.json`. The scattering convention is

```text
scatter[region, destination_group, source_group]
```

so rows identify destination groups and columns identify source groups. Group numbering in plot labels starts at 1.

Two bookkeeping details are worth noting. With `multigroup_warm_start=False`, every group solve restarts from a unit angular flux; setting it to `True` reuses each group's preceding angular flux and usually reduces work. Scattering data are always selected from the actual spatial region.

In [ ]:
multigroup_settings = SNSettings(
    order=2,
    inner_tolerance=1.0e-5,
    outer_tolerance=1.0e-4,
    eigenvalue_tolerance=1.0e-5,
    multigroup_warm_start=False,  # restart each group from a unit flux
)

multigroup_solver = SNSolver.from_case(
    DATA_DIRECTORY,
    case='homogeneous',       # also try 'reflected'
    multigroup=True,
    settings=multigroup_settings,
)
multigroup_result = multigroup_solver.solve_critical()

print(f'k_eff = {multigroup_result.keff:.8f}')
print(f'groups = {multigroup_result.scalar_flux.shape[0]}')
print(f'power iterations = {multigroup_result.power_iterations}')

In [ ]:
mesh = multigroup_result.problem.mesh
group_fluxes = multigroup_result.normalized_scalar_flux()

fig, ax = plt.subplots()
for group, flux in enumerate(group_fluxes, start=1):
    ax.plot(mesh.cell_centers, flux, label=f'group {group}')
for interface in mesh.region_edges[1:-1]:
    ax.axvline(interface, color='0.7', ls='--', lw=1)
ax.set(xlabel='$z$', ylabel='scalar flux / global maximum',
       title=f'Multigroup solution, $k_{{eff}}={multigroup_result.keff:.7f}$')
ax.legend();

### Inspecting the nested multigroup convergence

The following selects the first power iteration, lists every outer iteration, and reports both the group-flux changes and the number of inner iterations.

In [ ]:
first_power = multigroup_result.history[0]
print(f'Power iteration {first_power.iteration}:')
for outer_pass in first_power.outer_iterations:
    counts = [history.iterations for history in outer_pass.inner_iterations]
    errors = [f'{error:.2e}' for error in outer_pass.group_flux_errors]
    print(f'  outer iteration {outer_pass.iteration}: errors={errors}, inner iterations={counts}')

## Fixed-source calculation

An angle-integrated isotropic source can be assigned to a spatial interval with `isotropic_source`. The example below reproduces the MATLAB workflow: it first calculates the unscaled critical eigenvalue and then scales `nu*Sigma_f` so that the multiplying system has the requested subcritical value $k_{eff}=0.95$. Omit `target_keff` for a nonmultiplying source calculation, or pass `fission_scale=1.0` to use the tabulated fission production directly.

In [ ]:
fixed_source_settings = SNSettings(
    order=2,
    inner_tolerance=1.0e-6,
    eigenvalue_tolerance=1.0e-6,
)
source_solver = SNSolver.from_case(
    DATA_DIRECTORY, 'homogeneous', multigroup=False, settings=fixed_source_settings
)

external_source = source_solver.isotropic_source(
    left=30.0, right=70.0, strength=1.0, group=0
)
fixed_source_result = source_solver.solve_fixed_source(
    external_source, target_keff=0.95
)

print(f'reference k_eff = {fixed_source_result.reference_keff:.8f}')
print(f'fission scale = {fixed_source_result.fission_scale:.8f}')
print(f'inner iterations = {fixed_source_result.inner_iterations}')
print(f'converged = {fixed_source_result.converged}')

In [ ]:
mesh = fixed_source_result.problem.mesh
flux = fixed_source_result.scalar_flux[0]

fig, ax = plt.subplots()
ax.plot(mesh.cell_centers, flux, lw=2, label='scalar flux')
ax.axvspan(30.0, 70.0, color='tab:orange', alpha=0.2, label='external source')
ax.set(xlabel='$z$', ylabel='scalar flux', title='Fixed-source solution')
ax.legend();

## Suggested experiments

- Change `case` between `homogeneous`, `reflected`, and (for one group) `5regions`.
- Compare $S_2$, $S_4$, and $S_8$.
- Tighten one tolerance at a time and inspect which iteration level becomes more expensive.
- Examine `result.angular_flux[group, direction, cell]` and `result.directions`.
- Inspect material data through `solver.problem.cross_sections`.

The solver code is in `sn_1d.py`; its small methods follow the iteration hierarchy shown at the top of this notebook.